##SPARQL Query Analysis

This notebook fetches SPARQL queries from the LSQ endpoint, analyzes duplicates, queries the DBpedia endpoint, and saves successful queries for further analysis.

In [1]:
import os
import time
import json
import pandas as pd
import requests
from tqdm import tqdm
from SPARQLWrapper import SPARQLWrapper, JSON
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [2]:

# ---------- CONFIG ----------
LSQ_ENDPOINT = "http://lsq.aksw.org/sparql"
BATCH_SIZE = 1000
NUM_BATCHES = 300           # reduce for quick sampled runs
SAMPLE_SIZE = None         # set to int to cap total queries after fetch (None = no cap)
OUTPUT_FILE = "successful_queries_wikidata.json"
MIN_RESULTS_PER_QUERY = 1
MAX_BINDINGS_PER_QUERY = 50
REQUEST_TIMEOUT = 15
DELAY_BETWEEN_REQUESTS = 0.1
SAVE_EVERY = 50


In [3]:
WIKIDATA_ENDPOINT = "https://wikidata.data.dice-research.org/sparql"
QUERY_TEMPLATE_WIKIDATA = """
PREFIX lsqv: <http://lsq.aksw.org/vocab#>
PREFIX sd:   <http://www.w3.org/ns/sparql-service-description#>

SELECT DISTINCT ?text
FROM <http://lsq.aksw.org/wikidata>
WHERE {{
  ?query lsqv:text ?text .
  ?query lsqv:hasSpin ?spin .
  ?spin a ?spinType .
  ?query lsqv:hasStructuralFeatures ?features .
}}
LIMIT {limit}
OFFSET {offset}
"""

def fetch_select_queries_lsqs(
    num_batches=NUM_BATCHES,
    batch_size=BATCH_SIZE,
    sample_size=SAMPLE_SIZE,
    delay=0.5,
    timeout=120
):
    all_queries = []

    for i in range(num_batches):
        offset = i * batch_size
        print(f"Fetching batch {i+1}/{num_batches}, OFFSET {offset}")

        try:
            sparql = SPARQLWrapper(LSQ_ENDPOINT)
            sparql.setQuery(
                QUERY_TEMPLATE_WIKIDATA.format(
                    limit=batch_size,
                    offset=offset
                )
            )
            sparql.setReturnFormat(JSON)
            sparql.setTimeout(timeout)

            results = sparql.query().convert()
            texts = [
                b["text"]["value"]
                for b in results.get("results", {}).get("bindings", [])
            ]

            if not texts:
                print("No results returned — stopping early.")
                break

            all_queries.extend(texts)
            print(f"Fetched {len(texts)} queries")

        except Exception as e:
            print(f"LSQ fetch error at OFFSET {offset}: {e}")

        time.sleep(delay)

    if sample_size:
        all_queries = all_queries[:sample_size]

    return pd.DataFrame({"query": all_queries})

In [4]:
# Uncomment to run fetch
df = fetch_select_queries_lsqs()
df.to_csv("sampled_select_queries.csv", index=False)

Fetching batch 1/3, OFFSET 0
Fetched 10 queries
Fetching batch 2/3, OFFSET 10
Fetched 10 queries
Fetching batch 3/3, OFFSET 20
Fetched 10 queries


In [5]:
# ---------- Load sampled queries ----------
if not os.path.exists("sampled_select_queries.csv"):
    raise FileNotFoundError("Run the LSQ fetch step first and save sampled_select_queries.csv")

df = pd.read_csv("sampled_select_queries.csv")
print(f"Loaded {len(df)} sampled queries")
df_unique = df.drop_duplicates(subset="query").reset_index(drop=True)
print(f"Unique queries after dedup: {len(df_unique)}")


Loaded 30 sampled queries
Unique queries after dedup: 30


In [6]:
# ---------- Prepare HTTP session with retries ----------
session = requests.Session()
retries = Retry(total=3, backoff_factor=1, status_forcelist=(429,500,502,503,504))
session.mount("https://", HTTPAdapter(max_retries=retries))
HEADERS = {"Accept": "application/sparql-results+json"}

def query_dbpedia(q):
    try:
        r = session.get(WIKIDATA_ENDPOINT, params={"query": q}, headers=HEADERS, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        return {"_error": str(e)}

In [7]:
# ---------- Run queries and collect successful ones (sample-friendly) ----------
successful = []
for i, row in tqdm(df_unique.iterrows(), total=len(df_unique), desc="Querying DBpedia"):
    sparql = str(row["query"]).strip()
    if not sparql:
        continue
    res = query_dbpedia(sparql)
    if "_error" in res:
        # skip / optionally log
        continue
    bindings = res.get("results", {}).get("bindings", []) or []
    if len(bindings) >= MIN_RESULTS_PER_QUERY:
        successful.append({
            "sparql_query": sparql,
            "results_count": len(bindings),
            "results": bindings[:MAX_BINDINGS_PER_QUERY]
        })
    time.sleep(DELAY_BETWEEN_REQUESTS)
    if SAVE_EVERY and len(successful) and len(successful) % SAVE_EVERY == 0:
        with open(OUTPUT_FILE + ".partial", "w", encoding="utf-8") as f:
            json.dump(successful, f, ensure_ascii=False, indent=2)


Querying DBpedia: 100%|██████████| 30/30 [00:25<00:00,  1.16it/s]


In [8]:
# ---------- Final save ----------
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(successful, f, ensure_ascii=False, indent=2)

print(f"Saved {len(successful)} successful queries to {OUTPUT_FILE}")

Saved 9 successful queries to successful_queries_wikidata.json
